# 1. Imports

In [17]:
import os
import gc
import pickle
import warnings
import numpy as np
import scipy.io
from scipy.signal import butter, filtfilt, welch, resample

from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

warnings.filterwarnings("ignore")

## 1.1 Install and import kagglehub

In [18]:
!pip install -q kagglehub

In [19]:
import kagglehub

# 2. Helper functions for preprocessing

In [20]:
def bandpass_filter(data, lowcut=1, highcut=45, fs=128, order=4):
    """
    data shape: (channels, samples)
    """
    nyquist = 0.5 * fs
    low = lowcut / nyquist
    high = highcut / nyquist
    b, a = butter(order, [low, high], btype='band')
    return filtfilt(b, a, data, axis=1)


def resample_signal(data, orig_fs=128, target_fs=256):
    """
    data shape: (channels, samples)
    """
    if orig_fs == target_fs:
        return data

    num_samples = int(data.shape[1] * target_fs / orig_fs)
    return resample(data, num_samples, axis=1)


def segment_signal(data, window_size=1024, step_size=512):
    segments = []
    n_samples = data.shape[1]

    for start in range(0, n_samples - window_size + 1, step_size):
        end = start + window_size
        segment = data[:, start:end]
        segments.append(segment)

    return np.array(segments)


def zscore_with_stats(signal, mean, std):
    return (signal - mean) / (std + 1e-8)


def compute_subject_stats(trials):
    all_data = np.concatenate(trials, axis=1)
    mean = np.mean(all_data, axis=1, keepdims=True)
    std = np.std(all_data, axis=1, keepdims=True)
    return mean, std

# 3. Helper functions for feature extraction (Bandpower)

In [21]:
def extract_bandpower(segment, fs=256):
    """
    segment shape: (channels, samples)
    returns feature vector shape: (channels * 5,)
    """
    bands = {
        "delta": (1, 4),
        "theta": (4, 8),
        "alpha": (8, 13),
        "beta":  (13, 30),
        "gamma": (30, 45)
    }

    features = []

    for ch in range(segment.shape[0]):
        freqs, psd = welch(segment[ch], fs=fs, nperseg=min(256, segment.shape[1]))

        for low, high in bands.values():
            idx = np.logical_and(freqs >= low, freqs < high)
            bandpower = np.trapz(psd[idx], freqs[idx]) if np.any(idx) else 0.0
            features.append(bandpower)

    return np.array(features, dtype=np.float32)


def extract_features_from_segments(X_segments, fs=256):
    """
    X_segments shape: (num_segments, channels, samples)
    returns shape: (num_segments, num_features)
    """
    features = [extract_bandpower(segment, fs=fs) for segment in X_segments]
    return np.array(features, dtype=np.float32)

# 4. Download and load DEAP

In [22]:
path_deap = kagglehub.dataset_download("manh123df/deap-dataset")
print("DEAP path:", path_deap)

DEAP path: /kaggle/input/datasets/manh123df/deap-dataset


In [23]:
deap_dir = os.path.join(path_deap, "deap-dataset", "data_preprocessed_python")
print("DEAP dir:", deap_dir)
print("Exists:", os.path.exists(deap_dir))
print("Example files:", os.listdir(deap_dir)[:5])

DEAP dir: /kaggle/input/datasets/manh123df/deap-dataset/deap-dataset/data_preprocessed_python
Exists: True
Example files: ['s20.dat', 's17.dat', 's31.dat', 's14.dat', 's32.dat']


In [24]:
deap_files = sorted([
    os.path.join(deap_dir, f)
    for f in os.listdir(deap_dir)
    if f.endswith(".dat")
])

print("Number of DEAP subject files:", len(deap_files))
print(deap_files[:3])

Number of DEAP subject files: 32
['/kaggle/input/datasets/manh123df/deap-dataset/deap-dataset/data_preprocessed_python/s01.dat', '/kaggle/input/datasets/manh123df/deap-dataset/deap-dataset/data_preprocessed_python/s02.dat', '/kaggle/input/datasets/manh123df/deap-dataset/deap-dataset/data_preprocessed_python/s03.dat']


# 5. Download and load DREAMER

In [25]:
path_dreamer = kagglehub.dataset_download("phhasian0710/dreamer")
print("DREAMER path:", path_dreamer)

DREAMER path: /kaggle/input/datasets/phhasian0710/dreamer


In [26]:
print(os.listdir(path_dreamer))

['DREAMER.mat', 'dreamer']


In [27]:
dreamer_file = os.path.join(path_dreamer, "DREAMER.mat")
print("DREAMER file:", dreamer_file)

DREAMER file: /kaggle/input/datasets/phhasian0710/dreamer/DREAMER.mat


In [28]:
dreamer_data = scipy.io.loadmat(dreamer_file)
print(dreamer_data.keys())

dict_keys(['__header__', '__version__', '__globals__', 'DREAMER'])


In [29]:
dreamer_main = dreamer_data["DREAMER"][0, 0]
dreamer_subjects = dreamer_main["Data"]

print("EEG sampling rate:", dreamer_main["EEG_SamplingRate"])
print("Number of subjects:", dreamer_main["noOfSubjects"])
print("Number of video sequences:", dreamer_main["noOfVideoSequences"])
print("dreamer_subjects shape:", dreamer_subjects.shape)

EEG sampling rate: [[128]]
Number of subjects: [[23]]
Number of video sequences: [[18]]
dreamer_subjects shape: (1, 23)


# 6. Define channel names and common channels

In [30]:
# DEAP 32 EEG channels
DEAP_CHANNELS = [
    "Fp1", "AF3", "F3", "F7", "FC5", "FC1", "C3", "T7",
    "CP5", "CP1", "P3", "P7", "PO3", "O1", "Oz", "Pz",
    "Fp2", "AF4", "Fz", "F4", "F8", "FC6", "FC2", "Cz",
    "C4", "T8", "CP6", "CP2", "P4", "P8", "PO4", "O2"
]

# DREAMER 14 EEG channels
DREAMER_CHANNELS = [
    "AF3", "F7", "F3", "FC5", "T7", "P7", "O1",
    "O2", "P8", "T8", "FC6", "F4", "F8", "AF4"
]

# We use only the common channels, in the exact same order for both datasets
COMMON_CHANNELS = [
    "AF3", "F7", "F3", "FC5", "T7", "P7", "O1",
    "O2", "P8", "T8", "FC6", "F4", "F8", "AF4"
]

print("Number of common channels:", len(COMMON_CHANNELS))
print(COMMON_CHANNELS)

Number of common channels: 14
['AF3', 'F7', 'F3', 'FC5', 'T7', 'P7', 'O1', 'O2', 'P8', 'T8', 'FC6', 'F4', 'F8', 'AF4']


# 7. Channel index helper

In [31]:
def get_channel_indices(all_channels, selected_channels):
    indices = []
    for ch in selected_channels:
        if ch not in all_channels:
            raise ValueError(f"Channel {ch} not found.")
        indices.append(all_channels.index(ch))
    return indices


deap_selected_indices = get_channel_indices(DEAP_CHANNELS, COMMON_CHANNELS)
dreamer_selected_indices = get_channel_indices(DREAMER_CHANNELS, COMMON_CHANNELS)

print("DEAP selected indices:", deap_selected_indices)
print("DREAMER selected indices:", dreamer_selected_indices)

DEAP selected indices: [1, 3, 2, 4, 7, 11, 13, 31, 29, 25, 21, 19, 20, 17]
DREAMER selected indices: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]


# 8. Set label type

In [32]:
LABEL_TYPE = "valence"   # change to "arousal" if needed
print("Current label type:", LABEL_TYPE)

Current label type: valence


# 9. DEAP subject loader

In [33]:
def load_deap_subject(subject_path, selected_indices, label_type="valence", label_threshold=5.0):
    """
    subject_path: path to one DEAP .dat file

    returns:
        X_subject: (num_segments, n_channels, 1024)
        y_subject: (num_segments,)
    """
    with open(subject_path, "rb") as f:
        subject = pickle.load(f, encoding="latin1")

    data = subject["data"]      # shape: (40, 40, 8064)
    labels = subject["labels"]  # shape: (40, 4)

    processed_trials = []
    trial_labels = []

    if label_type == "valence":
        label_col = 0
    elif label_type == "arousal":
        label_col = 1
    else:
        raise ValueError("label_type must be 'valence' or 'arousal'")

    for i in range(data.shape[0]):
        # First 32 channels are EEG
        trial = data[i, :32, :]

        # Keep only common channels
        trial = trial[selected_indices, :]

        # Remove first 3 seconds baseline (384 samples at 128 Hz)
        trial = trial[:, 384:]

        # Preprocessing
        trial = bandpass_filter(trial, lowcut=1, highcut=45, fs=128, order=4)
        trial = resample_signal(trial, orig_fs=128, target_fs=256)

        processed_trials.append(trial)

        score = labels[i, label_col]
        label = 1 if score > label_threshold else 0
        trial_labels.append(label)

    # Subject-wise normalization
    mean, std = compute_subject_stats(processed_trials)

    X_subject = []
    y_subject = []

    for trial, label in zip(processed_trials, trial_labels):
        trial_norm = zscore_with_stats(trial, mean, std)
        segments = segment_signal(trial_norm, window_size=1024, step_size=512)

        X_subject.append(segments)
        y_subject.extend([label] * len(segments))

    X_subject = np.vstack(X_subject).astype(np.float32)
    y_subject = np.array(y_subject, dtype=np.int64)

    return X_subject, y_subject

# 10. DREAMER subject loader

In [34]:
def load_dreamer_subject(subject_data, selected_indices, label_type="valence", label_threshold=3.0):
    """
    subject_data: one DREAMER subject object from dreamer_subjects[0, subject_id]

    returns:
        X_subject: (num_segments, n_channels, 1024)
        y_subject: (num_segments,)
    """
    subject_eeg = subject_data["EEG"][0, 0]["stimuli"][0, 0]

    if label_type == "valence":
        subject_scores = subject_data["ScoreValence"][0, 0].flatten()
    elif label_type == "arousal":
        subject_scores = subject_data["ScoreArousal"][0, 0].flatten()
    else:
        raise ValueError("label_type must be 'valence' or 'arousal'")

    processed_trials = []
    trial_labels = []

    for i in range(subject_eeg.shape[0]):
        trial = subject_eeg[i, 0].T   # shape: (14, samples)

        # Keep only common channels
        trial = trial[selected_indices, :]

        # Preprocessing
        trial = bandpass_filter(trial, lowcut=1, highcut=45, fs=128, order=4)
        trial = resample_signal(trial, orig_fs=128, target_fs=256)

        processed_trials.append(trial)

        label = 1 if subject_scores[i] > label_threshold else 0
        trial_labels.append(label)

    # Subject-wise normalization
    mean, std = compute_subject_stats(processed_trials)

    X_subject = []
    y_subject = []

    for trial, label in zip(processed_trials, trial_labels):
        trial_norm = zscore_with_stats(trial, mean, std)
        segments = segment_signal(trial_norm, window_size=1024, step_size=512)

        X_subject.append(segments)
        y_subject.extend([label] * len(segments))

    X_subject = np.vstack(X_subject).astype(np.float32)
    y_subject = np.array(y_subject, dtype=np.int64)

    return X_subject, y_subject

# 11.Quick test on one DEAP subject

In [35]:
X_segments_sub1_deap, y_labels_sub1_deap = load_deap_subject(
    deap_files[0],
    selected_indices=deap_selected_indices,
    label_type=LABEL_TYPE,
    label_threshold=5.0
)

print("DEAP subject 1 segments shape:", X_segments_sub1_deap.shape)
print("DEAP subject 1 labels shape:", y_labels_sub1_deap.shape)
print("Unique labels:", np.unique(y_labels_sub1_deap))
print("Class counts:", np.bincount(y_labels_sub1_deap))

DEAP subject 1 segments shape: (1160, 14, 1024)
DEAP subject 1 labels shape: (1160,)
Unique labels: [0 1]
Class counts: [986 174]


# 12. Quick test on one DREAMER subject

In [36]:
subject_1 = dreamer_subjects[0, 0]

X_segments_sub1_dreamer, y_labels_sub1_dreamer = load_dreamer_subject(
    subject_1,
    selected_indices=dreamer_selected_indices,
    label_type=LABEL_TYPE,
    label_threshold=3.0
)

print("DREAMER subject 1 segments shape:", X_segments_sub1_dreamer.shape)
print("DREAMER subject 1 labels shape:", y_labels_sub1_dreamer.shape)
print("Unique labels:", np.unique(y_labels_sub1_dreamer))
print("Class counts:", np.bincount(y_labels_sub1_dreamer))

DREAMER subject 1 segments shape: (1843, 14, 1024)
DREAMER subject 1 labels shape: (1843,)
Unique labels: [0 1]
Class counts: [977 866]


# 13. Build full DEAP dataset

In [37]:
X_deap = []
y_deap = []

num_subjects_deap = len(deap_files)

for subject_id, subject_path in enumerate(deap_files):
    X_sub, y_sub = load_deap_subject(
        subject_path=subject_path,
        selected_indices=deap_selected_indices,
        label_type=LABEL_TYPE,
        label_threshold=5.0
    )

    X_deap.append(X_sub)
    y_deap.append(y_sub)

    print(f"DEAP Subject {subject_id+1}: segments={X_sub.shape[0]}, class counts={np.bincount(y_sub)}")

X_deap = np.vstack(X_deap)
y_deap = np.concatenate(y_deap)

print("\nFinal DEAP shapes:")
print("X_deap:", X_deap.shape)
print("y_deap:", y_deap.shape)
print("Unique labels:", np.unique(y_deap))
print("Class counts:", np.bincount(y_deap))

DEAP Subject 1: segments=1160, class counts=[986 174]
DEAP Subject 2: segments=1160, class counts=[986 174]
DEAP Subject 3: segments=1160, class counts=[551 609]
DEAP Subject 4: segments=1160, class counts=[1044  116]
DEAP Subject 5: segments=1160, class counts=[841 319]
DEAP Subject 6: segments=1160, class counts=[667 493]
DEAP Subject 7: segments=1160, class counts=[870 290]
DEAP Subject 8: segments=1160, class counts=[928 232]
DEAP Subject 9: segments=1160, class counts=[1015  145]
DEAP Subject 10: segments=1160, class counts=[899 261]
DEAP Subject 11: segments=1160, class counts=[725 435]
DEAP Subject 12: segments=1160, class counts=[1102   58]
DEAP Subject 13: segments=1160, class counts=[1102   58]
DEAP Subject 14: segments=1160, class counts=[928 232]
DEAP Subject 15: segments=1160, class counts=[812 348]
DEAP Subject 16: segments=1160, class counts=[986 174]
DEAP Subject 17: segments=1160, class counts=[899 261]
DEAP Subject 18: segments=1160, class counts=[957 203]
DEAP Subjec

# 14. Build full DREAMER dataset

In [38]:
X_dreamer = []
y_dreamer = []

num_subjects_dreamer = dreamer_subjects.shape[1]

for subject_id in range(num_subjects_dreamer):
    subject = dreamer_subjects[0, subject_id]

    X_sub, y_sub = load_dreamer_subject(
        subject_data=subject,
        selected_indices=dreamer_selected_indices,
        label_type=LABEL_TYPE,
        label_threshold=3.0
    )

    X_dreamer.append(X_sub)
    y_dreamer.append(y_sub)

    print(f"DREAMER Subject {subject_id+1}: segments={X_sub.shape[0]}, class counts={np.bincount(y_sub)}")

X_dreamer = np.vstack(X_dreamer)
y_dreamer = np.concatenate(y_dreamer)

print("\nFinal DREAMER shapes:")
print("X_dreamer:", X_dreamer.shape)
print("y_dreamer:", y_dreamer.shape)
print("Unique labels:", np.unique(y_dreamer))
print("Class counts:", np.bincount(y_dreamer))

DREAMER Subject 1: segments=1843, class counts=[977 866]
DREAMER Subject 2: segments=1843, class counts=[1256  587]
DREAMER Subject 3: segments=1843, class counts=[943 900]
DREAMER Subject 4: segments=1843, class counts=[1414  429]
DREAMER Subject 5: segments=1843, class counts=[1158  685]
DREAMER Subject 6: segments=1843, class counts=[1001  842]
DREAMER Subject 7: segments=1843, class counts=[1207  636]
DREAMER Subject 8: segments=1843, class counts=[1218  625]
DREAMER Subject 9: segments=1843, class counts=[1029  814]
DREAMER Subject 10: segments=1843, class counts=[ 695 1148]
DREAMER Subject 11: segments=1843, class counts=[1189  654]
DREAMER Subject 12: segments=1843, class counts=[901 942]
DREAMER Subject 13: segments=1843, class counts=[ 804 1039]
DREAMER Subject 14: segments=1843, class counts=[1024  819]
DREAMER Subject 15: segments=1843, class counts=[1207  636]
DREAMER Subject 16: segments=1843, class counts=[1160  683]
DREAMER Subject 17: segments=1843, class counts=[1192  

# 15. Extract bandpower features

In [39]:
print("Extracting DEAP features...")
X_deap_features = extract_features_from_segments(X_deap, fs=256)

print("Extracting DREAMER features...")
X_dreamer_features = extract_features_from_segments(X_dreamer, fs=256)

print("\nFeature shapes:")
print("X_deap_features:", X_deap_features.shape)
print("X_dreamer_features:", X_dreamer_features.shape)

Extracting DEAP features...
Extracting DREAMER features...

Feature shapes:
X_deap_features: (37120, 70)
X_dreamer_features: (42389, 70)


## 15.1 Save extracted features and labels

In [47]:
np.savez_compressed(
    "cross_dataset_svm_features.npz",
    X_deap_features=X_deap_features,
    y_deap=y_deap,
    X_dreamer_features=X_dreamer_features,
    y_dreamer=y_dreamer,
    label_type=LABEL_TYPE
)

## 15.2 Load

In [46]:
data = np.load("cross_dataset_svm_features.npz")

X_deap_features = data["X_deap_features"]
y_deap = data["y_deap"]
X_dreamer_features = data["X_dreamer_features"]
y_dreamer = data["y_dreamer"]

print("Loaded all arrays successfully.")
print("X_deap_features:", X_deap_features.shape)
print("y_deap:", y_deap.shape)
print("X_dreamer_features:", X_dreamer_features.shape)
print("y_dreamer:", y_dreamer.shape)

Loaded all arrays successfully.
X_deap_features: (37120, 70)
y_deap: (37120,)
X_dreamer_features: (42389, 70)
y_dreamer: (42389,)


# 16. Sanity check

In [40]:
print("DEAP number of features per segment:", X_deap_features.shape[1])
print("DREAMER number of features per segment:", X_dreamer_features.shape[1])

assert X_deap_features.shape[1] == X_dreamer_features.shape[1], \
    "Feature dimensions do not match between DEAP and DREAMER."

print("Feature dimensions match correctly.")

DEAP number of features per segment: 70
DREAMER number of features per segment: 70
Feature dimensions match correctly.


In [41]:
print(np.bincount(y_deap))
print(np.bincount(y_dreamer))

[29319  7801]
[24961 17428]


# 17. Cross-dataset SVM function

In [48]:
def run_cross_dataset_svm(X_train, y_train, X_test, y_test, train_name="Train", test_name="Test"):
    print(f"\n========== TRAIN ON {train_name} | TEST ON {test_name} ==========")
    print("X_train shape:", X_train.shape)
    print("y_train shape:", y_train.shape)
    print("X_test shape:", X_test.shape)
    print("y_test shape:", y_test.shape)
    print("Train class counts:", np.bincount(y_train))
    print("Test class counts:", np.bincount(y_test))

    # Standardize using train set only
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # SVM model
    model = SVC(
        kernel="rbf",
        C=1.0,
        gamma="scale",
        class_weight="balanced"
    )

    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    macro_f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)
    cm = confusion_matrix(y_test, y_pred)

    print("\nResults:")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-score:  {f1:.4f}")
    print(f"Macro F1:  {macro_f1:.4f}")

    print("\nConfusion Matrix:")
    print(cm)

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, digits=4, zero_division=0))

    return {
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "macro_f1": macro_f1,
        "confusion_matrix": cm,
        "y_pred": y_pred
    }

# 18. Experiment 1 — Train on DEAP, test on DREAMER

In [49]:
results_deap_to_dreamer = run_cross_dataset_svm(
    X_train=X_deap_features,
    y_train=y_deap,
    X_test=X_dreamer_features,
    y_test=y_dreamer,
    train_name="DEAP",
    test_name="DREAMER"
)


========== TRAIN ON DEAP | TEST ON DREAMER ==========
X_train shape: (37120, 70)
y_train shape: (37120,)
X_test shape: (42389, 70)
y_test shape: (42389,)
Train class counts: [29319  7801]
Test class counts: [24961 17428]

Results:
Accuracy:  0.5877
Precision: 0.4327
Recall:    0.0094
F1-score:  0.0184
Macro F1:  0.3787

Confusion Matrix:
[[24746   215]
 [17264   164]]

Classification Report:
              precision    recall  f1-score   support

           0     0.5891    0.9914    0.7390     24961
           1     0.4327    0.0094    0.0184     17428

    accuracy                         0.5877     42389
   macro avg     0.5109    0.5004    0.3787     42389
weighted avg     0.5248    0.5877    0.4427     42389



# 19. Experiment 2 — Train on DREAMER, test on DEAP

In [50]:
results_dreamer_to_deap = run_cross_dataset_svm(
    X_train=X_dreamer_features,
    y_train=y_dreamer,
    X_test=X_deap_features,
    y_test=y_deap,
    train_name="DREAMER",
    test_name="DEAP"
)


========== TRAIN ON DREAMER | TEST ON DEAP ==========
X_train shape: (42389, 70)
y_train shape: (42389,)
X_test shape: (37120, 70)
y_test shape: (37120,)
Train class counts: [24961 17428]
Test class counts: [29319  7801]

Results:
Accuracy:  0.7418
Precision: 0.2358
Recall:    0.1019
F1-score:  0.1423
Macro F1:  0.4952

Confusion Matrix:
[[26742  2577]
 [ 7006   795]]

Classification Report:
              precision    recall  f1-score   support

           0     0.7924    0.9121    0.8481     29319
           1     0.2358    0.1019    0.1423      7801

    accuracy                         0.7418     37120
   macro avg     0.5141    0.5070    0.4952     37120
weighted avg     0.6754    0.7418    0.6997     37120



# 20. Save final SVM results

In [51]:
cross_dataset_results = {
    "label_type": LABEL_TYPE,
    "deap_to_dreamer": {
        "accuracy": results_deap_to_dreamer["accuracy"],
        "precision": results_deap_to_dreamer["precision"],
        "recall": results_deap_to_dreamer["recall"],
        "f1": results_deap_to_dreamer["f1"],
        "macro_f1": results_deap_to_dreamer["macro_f1"],
        "confusion_matrix": results_deap_to_dreamer["confusion_matrix"]
    },
    "dreamer_to_deap": {
        "accuracy": results_dreamer_to_deap["accuracy"],
        "precision": results_dreamer_to_deap["precision"],
        "recall": results_dreamer_to_deap["recall"],
        "f1": results_dreamer_to_deap["f1"],
        "macro_f1": results_dreamer_to_deap["macro_f1"],
        "confusion_matrix": results_dreamer_to_deap["confusion_matrix"]
    }
}

np.save("cross_dataset_svm_results.npy", cross_dataset_results, allow_pickle=True)
print("Cross-dataset SVM results saved to cross_dataset_svm_results.npy")
print(cross_dataset_results)

Cross-dataset SVM results saved to cross_dataset_svm_results.npy
{'label_type': 'valence', 'deap_to_dreamer': {'accuracy': 0.5876524570053552, 'precision': 0.43271767810026385, 'recall': 0.009410144594904751, 'f1': 0.018419722581007467, 'macro_f1': 0.3787130791011979, 'confusion_matrix': array([[24746,   215],
       [17264,   164]])}, 'dreamer_to_deap': {'accuracy': 0.7418372844827587, 'precision': 0.23576512455516013, 'recall': 0.10191001153698244, 'f1': 0.14230734807124318, 'macro_f1': 0.495178917031166, 'confusion_matrix': array([[26742,  2577],
       [ 7006,   795]])}}


In [52]:
print(X_deap.shape)
print(X_dreamer.shape)

(37120, 14, 1024)
(42389, 14, 1024)


# 21.Imports for EEGNet

In [54]:
import tensorflow as tf
from tensorflow.keras import Model
from tensorflow.keras.constraints import max_norm
from tensorflow.keras.layers import (
    Input, Conv2D, BatchNormalization, DepthwiseConv2D,
    AveragePooling2D, SeparableConv2D, Dropout,
    SpatialDropout2D, Activation, Flatten, Dense
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.utils.class_weight import compute_class_weight

# 22. Prepare input shape for EEGNet

In [55]:
X_deap_eegnet = X_deap[..., np.newaxis].astype(np.float32)
X_dreamer_eegnet = X_dreamer[..., np.newaxis].astype(np.float32)

print("X_deap_eegnet:", X_deap_eegnet.shape)
print("X_dreamer_eegnet:", X_dreamer_eegnet.shape)

X_deap_eegnet: (37120, 14, 1024, 1)
X_dreamer_eegnet: (42389, 14, 1024, 1)


# 23. EEGNet model

In [56]:
def build_eegnet(nb_classes=2, Chans=14, Samples=1024,
                 dropoutRate=0.5, kernLength=64, F1=8,
                 D=2, F2=16, norm_rate=0.25, dropoutType='Dropout'):

    if dropoutType == 'SpatialDropout2D':
        dropoutType = SpatialDropout2D
    else:
        dropoutType = Dropout

    input1 = Input(shape=(Chans, Samples, 1))

    block1 = Conv2D(
        F1, (1, kernLength),
        padding='same',
        use_bias=False
    )(input1)
    block1 = BatchNormalization()(block1)

    block1 = DepthwiseConv2D(
        (Chans, 1),
        use_bias=False,
        depth_multiplier=D,
        depthwise_constraint=max_norm(1.)
    )(block1)
    block1 = BatchNormalization()(block1)
    block1 = Activation('elu')(block1)
    block1 = AveragePooling2D((1, 4))(block1)
    block1 = dropoutType(dropoutRate)(block1)

    block2 = SeparableConv2D(
        F2, (1, 16),
        use_bias=False,
        padding='same'
    )(block1)
    block2 = BatchNormalization()(block2)
    block2 = Activation('elu')(block2)
    block2 = AveragePooling2D((1, 8))(block2)
    block2 = dropoutType(dropoutRate)(block2)

    flatten = Flatten()(block2)

    dense = Dense(
        nb_classes,
        kernel_constraint=max_norm(norm_rate)
    )(flatten)
    softmax = Activation('softmax')(dense)

    return Model(inputs=input1, outputs=softmax)

# 24. Cross-dataset EEGNet function

In [58]:
def run_cross_dataset_eegnet(X_train, y_train, X_test, y_test,
                             train_name="Train", test_name="Test",
                             epochs=3, batch_size=64, learning_rate=1e-3):

    print(f"\n========== TRAIN ON {train_name} | TEST ON {test_name} ==========")
    print("X_train shape:", X_train.shape)
    print("y_train shape:", y_train.shape)
    print("X_test shape:", X_test.shape)
    print("y_test shape:", y_test.shape)
    print("Train class counts:", np.bincount(y_train))
    print("Test class counts:", np.bincount(y_test))

    model = build_eegnet(
        nb_classes=2,
        Chans=X_train.shape[1],
        Samples=X_train.shape[2],
        dropoutRate=0.5,
        kernLength=64,
        F1=8,
        D=2,
        F2=16,
        norm_rate=0.25,
        dropoutType='Dropout'
    )

    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    classes = np.unique(y_train)
    class_weights = compute_class_weight(
        class_weight='balanced',
        classes=classes,
        y=y_train
    )
    class_weight_dict = {int(c): float(w) for c, w in zip(classes, class_weights)}

    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=2,
        restore_best_weights=True
    )

    history = model.fit(
        X_train, y_train,
        validation_split=0.1,
        epochs=epochs,
        batch_size=batch_size,
        class_weight=class_weight_dict,
        callbacks=[early_stop],
        verbose=1
    )

    y_prob = model.predict(X_test, batch_size=batch_size, verbose=1)
    y_pred = np.argmax(y_prob, axis=1)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    macro_f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
    cm = confusion_matrix(y_test, y_pred)

    print("\nResults:")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-score:  {f1:.4f}")
    print(f"Macro F1:  {macro_f1:.4f}")

    print("\nConfusion Matrix:")
    print(cm)

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, digits=4, zero_division=0))

    return {
        "model": model,
        "history": history.history,
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "macro_f1": macro_f1,
        "confusion_matrix": cm,
        "y_pred": y_pred
    }

# 25. Experiment 1 — Train on DEAP, test on DREAMER

In [59]:
results_eegnet_deap_to_dreamer = run_cross_dataset_eegnet(
    X_train=X_deap_eegnet,
    y_train=y_deap,
    X_test=X_dreamer_eegnet,
    y_test=y_dreamer,
    train_name="DEAP",
    test_name="DREAMER",
    epochs=3,
    batch_size=64,
    learning_rate=1e-3
)


========== TRAIN ON DEAP | TEST ON DREAMER ==========
X_train shape: (37120, 14, 1024, 1)
y_train shape: (37120,)
X_test shape: (42389, 14, 1024, 1)
y_test shape: (42389,)
Train class counts: [29319  7801]
Test class counts: [24961 17428]


I0000 00:00:1776468393.428965      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1776468393.434627      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Epoch 1/3


I0000 00:00:1776468402.586606     282 service.cc:152] XLA service 0x79310802a510 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1776468402.586640     282 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1776468402.586644     282 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1776468403.000980     282 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1776468408.966514     282 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


522/522 ━━━━━━━━━━━━━━━━━━━━ 18s 17ms/step - accuracy: 0.5878 - loss: 0.6780 - val_accuracy: 0.5822 - val_loss: 0.6652
Epoch 2/3
522/522 ━━━━━━━━━━━━━━━━━━━━ 7s 14ms/step - accuracy: 0.6274 - loss: 0.6460 - val_accuracy: 0.5560 - val_loss: 0.6744
Epoch 3/3
522/522 ━━━━━━━━━━━━━━━━━━━━ 7s 14ms/step - accuracy: 0.6369 - loss: 0.6336 - val_accuracy: 0.6409 - val_loss: 0.6486
663/663 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step

Results:
Accuracy:  0.5774
Precision: 0.4189
Recall:    0.0721
F1-score:  0.1230
Macro F1:  0.4223

Confusion Matrix:
[[23219  1742]
 [16172  1256]]

Classification Report:
              precision    recall  f1-score   support

           0     0.5894    0.9302    0.7216     24961
           1     0.4189    0.0721    0.1230     17428

    accuracy                         0.5774     42389
   macro avg     0.5042    0.5011    0.4223     42389
weighted avg     0.5193    0.5774    0.4755     42389



# 26. Experiment 2 — Train on DREAMER, test on DEAP¶

In [60]:
results_eegnet_dreamer_to_deap = run_cross_dataset_eegnet(
    X_train=X_dreamer_eegnet,
    y_train=y_dreamer,
    X_test=X_deap_eegnet,
    y_test=y_deap,
    train_name="DREAMER",
    test_name="DEAP",
    epochs=3,
    batch_size=64,
    learning_rate=1e-3
)


========== TRAIN ON DREAMER | TEST ON DEAP ==========
X_train shape: (42389, 14, 1024, 1)
y_train shape: (42389,)
X_test shape: (37120, 14, 1024, 1)
y_test shape: (37120,)
Train class counts: [24961 17428]
Test class counts: [29319  7801]
Epoch 1/3
597/597 ━━━━━━━━━━━━━━━━━━━━ 21s 26ms/step - accuracy: 0.5097 - loss: 0.6961 - val_accuracy: 0.3843 - val_loss: 0.7355
Epoch 2/3
597/597 ━━━━━━━━━━━━━━━━━━━━ 8s 14ms/step - accuracy: 0.5792 - loss: 0.6659 - val_accuracy: 0.5303 - val_loss: 0.6916
Epoch 3/3
597/597 ━━━━━━━━━━━━━━━━━━━━ 8s 14ms/step - accuracy: 0.6119 - loss: 0.6458 - val_accuracy: 0.4978 - val_loss: 0.7173
580/580 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step

Results:
Accuracy:  0.6383
Precision: 0.1546
Recall:    0.1614
F1-score:  0.1579
Macro F1:  0.4638

Confusion Matrix:
[[22434  6885]
 [ 6542  1259]]

Classification Report:
              precision    recall  f1-score   support

           0     0.7742    0.7652    0.7697     29319
           1     0.1546    0.1614    0.1579      78

In [74]:
eegnet_cross_dataset_results = {
    "deap_to_dreamer": {
        "accuracy": results_eegnet_deap_to_dreamer["accuracy"],
        "precision": results_eegnet_deap_to_dreamer["precision"],
        "recall": results_eegnet_deap_to_dreamer["recall"],
        "f1": results_eegnet_deap_to_dreamer["f1"],
        "macro_f1": results_eegnet_deap_to_dreamer["macro_f1"],
        "confusion_matrix": results_eegnet_deap_to_dreamer["confusion_matrix"]
    },
    "dreamer_to_deap": {
        "accuracy": results_eegnet_dreamer_to_deap["accuracy"],
        "precision": results_eegnet_dreamer_to_deap["precision"],
        "recall": results_eegnet_dreamer_to_deap["recall"],
        "f1": results_eegnet_dreamer_to_deap["f1"],
        "macro_f1": results_eegnet_dreamer_to_deap["macro_f1"],
        "confusion_matrix": results_eegnet_dreamer_to_deap["confusion_matrix"]
    }
}

print(eegnet_cross_dataset_results)

{'deap_to_dreamer': {'accuracy': 0.5773903607067872, 'precision': 0.418945963975984, 'recall': 0.07206793665366078, 'f1': 0.1229805150298639, 'macro_f1': 0.42230266427773655, 'confusion_matrix': array([[23219,  1742],
       [16172,  1256]])}, 'dreamer_to_deap': {'accuracy': 0.63828125, 'precision': 0.15459233791748528, 'recall': 0.16138956544032818, 'f1': 0.15791784258388208, 'macro_f1': 0.46379467049856254, 'confusion_matrix': array([[22434,  6885],
       [ 6542,  1259]])}}


# 27. Imports for DeepConvNet

In [63]:
from tensorflow.keras.layers import MaxPooling2D, ELU

# 28. Build DeepConvNet

In [64]:
def build_deepconvnet(nb_classes=2, Chans=14, Samples=1024, dropoutRate=0.5):
    input_main = Input((Chans, Samples, 1))

    block1 = Conv2D(25, (1, 5), padding='same', use_bias=False)(input_main)
    block1 = Conv2D(25, (Chans, 1), use_bias=False)(block1)
    block1 = BatchNormalization()(block1)
    block1 = Activation('elu')(block1)
    block1 = MaxPooling2D((1, 2))(block1)
    block1 = Dropout(dropoutRate)(block1)

    block2 = Conv2D(50, (1, 5), padding='same', use_bias=False)(block1)
    block2 = BatchNormalization()(block2)
    block2 = Activation('elu')(block2)
    block2 = MaxPooling2D((1, 2))(block2)
    block2 = Dropout(dropoutRate)(block2)

    block3 = Conv2D(100, (1, 5), padding='same', use_bias=False)(block2)
    block3 = BatchNormalization()(block3)
    block3 = Activation('elu')(block3)
    block3 = MaxPooling2D((1, 2))(block3)
    block3 = Dropout(dropoutRate)(block3)

    block4 = Conv2D(200, (1, 5), padding='same', use_bias=False)(block3)
    block4 = BatchNormalization()(block4)
    block4 = Activation('elu')(block4)
    block4 = MaxPooling2D((1, 2))(block4)
    block4 = Dropout(dropoutRate)(block4)

    flatten = Flatten()(block4)
    dense = Dense(nb_classes, activation='softmax')(flatten)

    model = Model(inputs=input_main, outputs=dense)
    return model

# 29. Cross-dataset DeepConvNet function

In [65]:
def run_cross_dataset_deepconvnet(X_train, y_train, X_test, y_test,
                                  train_name="Train", test_name="Test",
                                  epochs=3, batch_size=64, learning_rate=1e-3):

    print(f"\n========== TRAIN ON {train_name} | TEST ON {test_name} ==========")
    print("X_train shape:", X_train.shape)
    print("y_train shape:", y_train.shape)
    print("X_test shape:", X_test.shape)
    print("y_test shape:", y_test.shape)
    print("Train class counts:", np.bincount(y_train))
    print("Test class counts:", np.bincount(y_test))

    model = build_deepconvnet(
        nb_classes=2,
        Chans=X_train.shape[1],
        Samples=X_train.shape[2],
        dropoutRate=0.5
    )

    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    classes = np.unique(y_train)
    class_weights = compute_class_weight(
        class_weight='balanced',
        classes=classes,
        y=y_train
    )
    class_weight_dict = {int(c): float(w) for c, w in zip(classes, class_weights)}

    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=2,
        restore_best_weights=True
    )

    history = model.fit(
        X_train, y_train,
        validation_split=0.1,
        epochs=epochs,
        batch_size=batch_size,
        class_weight=class_weight_dict,
        callbacks=[early_stop],
        verbose=1
    )

    y_prob = model.predict(X_test, batch_size=batch_size, verbose=1)
    y_pred = np.argmax(y_prob, axis=1)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    macro_f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
    cm = confusion_matrix(y_test, y_pred)

    print("\nResults:")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-score:  {f1:.4f}")
    print(f"Macro F1:  {macro_f1:.4f}")

    print("\nConfusion Matrix:")
    print(cm)

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, digits=4, zero_division=0))

    return {
        "model": model,
        "history": history.history,
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "macro_f1": macro_f1,
        "confusion_matrix": cm,
        "y_pred": y_pred
    }

# 30. Experiment 1 — Train on DEAP, test on DREAMER

In [66]:
results_deepconvnet_deap_to_dreamer = run_cross_dataset_deepconvnet(
    X_train=X_deap_eegnet,
    y_train=y_deap,
    X_test=X_dreamer_eegnet,
    y_test=y_dreamer,
    train_name="DEAP",
    test_name="DREAMER",
    epochs=3,
    batch_size=64,
    learning_rate=1e-3
)


========== TRAIN ON DEAP | TEST ON DREAMER ==========
X_train shape: (37120, 14, 1024, 1)
y_train shape: (37120,)
X_test shape: (42389, 14, 1024, 1)
y_test shape: (42389,)
Train class counts: [29319  7801]
Test class counts: [24961 17428]
Epoch 1/3


2026-04-17 23:51:31.888273: E external/local_xla/xla/service/slow_operation_alarm.cc:73] Trying algorithm eng3{k11=2} for conv %cudnn-conv.17 = (f32[100,200,1,5]{3,2,1,0}, u8[0]{0}) custom-call(f32[100,64,1,128]{3,2,1,0} %bitcast.16026, f32[200,64,1,128]{3,2,1,0} %bitcast.16033), window={size=1x128 pad=0_0x2_2}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convForward", metadata={op_type="Conv2DBackpropFilter" op_name="gradient_tape/functional_2_1/conv2d_6_1/convolution/Conv2DBackpropFilter" source_file="/usr/local/lib/python3.12/dist-packages/tensorflow/python/framework/ops.py" source_line=1200}, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"cudnn_conv_backend_config":{"conv_result_scale":1,"activation_mode":"kNone","side_input_scale":0,"leakyrelu_alpha":0},"force_earliest_schedule":false} is taking a while...
2026-04-17 23:51:32.605222: E external/local_xla/xla/service/slow_operation_alarm.cc:140] The operation took 1.717059525s
Trying algorithm e

522/522 ━━━━━━━━━━━━━━━━━━━━ 25s 26ms/step - accuracy: 0.5319 - loss: 1.3034 - val_accuracy: 0.3578 - val_loss: 1.0473
Epoch 2/3
522/522 ━━━━━━━━━━━━━━━━━━━━ 13s 24ms/step - accuracy: 0.5730 - loss: 0.9040 - val_accuracy: 0.3192 - val_loss: 1.0487
Epoch 3/3
522/522 ━━━━━━━━━━━━━━━━━━━━ 12s 24ms/step - accuracy: 0.5843 - loss: 0.8211 - val_accuracy: 0.3206 - val_loss: 0.8798
663/663 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step

Results:
Accuracy:  0.4625
Precision: 0.4037
Recall:    0.6443
F1-score:  0.4964
Macro F1:  0.4601

Confusion Matrix:
[[ 8377 16584]
 [ 6200 11228]]

Classification Report:
              precision    recall  f1-score   support

           0     0.5747    0.3356    0.4237     24961
           1     0.4037    0.6443    0.4964     17428

    accuracy                         0.4625     42389
   macro avg     0.4892    0.4899    0.4601     42389
weighted avg     0.5044    0.4625    0.4536     42389



# 31. Experiment 2 — Train on DREAMER, test on DEAP¶

In [67]:
results_deepconvnet_dreamer_to_deap = run_cross_dataset_deepconvnet(
    X_train=X_dreamer_eegnet,
    y_train=y_dreamer,
    X_test=X_deap_eegnet,
    y_test=y_deap,
    train_name="DREAMER",
    test_name="DEAP",
    epochs=3,
    batch_size=64,
    learning_rate=1e-3
)


========== TRAIN ON DREAMER | TEST ON DEAP ==========
X_train shape: (42389, 14, 1024, 1)
y_train shape: (42389,)
X_test shape: (37120, 14, 1024, 1)
y_test shape: (37120,)
Train class counts: [24961 17428]
Test class counts: [29319  7801]
Epoch 1/3
597/597 ━━━━━━━━━━━━━━━━━━━━ 29s 35ms/step - accuracy: 0.5004 - loss: 1.1957 - val_accuracy: 0.3874 - val_loss: 0.7366
Epoch 2/3
597/597 ━━━━━━━━━━━━━━━━━━━━ 14s 24ms/step - accuracy: 0.5363 - loss: 0.9626 - val_accuracy: 0.4225 - val_loss: 0.8060
Epoch 3/3
597/597 ━━━━━━━━━━━━━━━━━━━━ 14s 24ms/step - accuracy: 0.5681 - loss: 0.8487 - val_accuracy: 0.4779 - val_loss: 0.7277
580/580 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step

Results:
Accuracy:  0.6294
Precision: 0.1850
Recall:    0.2242
F1-score:  0.2027
Macro F1:  0.4807

Confusion Matrix:
[[21616  7703]
 [ 6052  1749]]

Classification Report:
              precision    recall  f1-score   support

           0     0.7813    0.7373    0.7586     29319
           1     0.1850    0.2242    0.2027      

In [75]:
deepconvnet_cross_dataset_results = {
    "deap_to_dreamer": {
        "accuracy": results_deepconvnet_deap_to_dreamer["accuracy"],
        "precision": results_deepconvnet_deap_to_dreamer["precision"],
        "recall": results_deepconvnet_deap_to_dreamer["recall"],
        "f1": results_deepconvnet_deap_to_dreamer["f1"],
        "macro_f1": results_deepconvnet_deap_to_dreamer["macro_f1"],
        "confusion_matrix": results_deepconvnet_deap_to_dreamer["confusion_matrix"]
    },
    "dreamer_to_deap": {
        "accuracy": results_deepconvnet_dreamer_to_deap["accuracy"],
        "precision": results_deepconvnet_dreamer_to_deap["precision"],
        "recall": results_deepconvnet_dreamer_to_deap["recall"],
        "f1": results_deepconvnet_dreamer_to_deap["f1"],
        "macro_f1": results_deepconvnet_dreamer_to_deap["macro_f1"],
        "confusion_matrix": results_deepconvnet_dreamer_to_deap["confusion_matrix"]
    }
}

print(deepconvnet_cross_dataset_results)

{'deap_to_dreamer': {'accuracy': 0.4625020642147727, 'precision': 0.403710628505681, 'recall': 0.644250631168235, 'f1': 0.49637488947833774, 'macro_f1': 0.4600595677600602, 'confusion_matrix': array([[ 8377, 16584],
       [ 6200, 11228]])}, 'dreamer_to_deap': {'accuracy': 0.6294450431034483, 'precision': 0.18504020313161235, 'recall': 0.22420202538136136, 'f1': 0.20274734828725438, 'macro_f1': 0.48068825466199105, 'confusion_matrix': array([[21616,  7703],
       [ 6052,  1749]])}}
